# REV1-01 — Event-Level Driver Analysis (v2)

Versi ini diperbaiki agar mengikuti struktur data yang **sudah ada** di komputer:

- Deteksi M2A final: `D:\ERA5_LAND\output_fd_03b_sensitivity`
- Driver pentad tahunan: `D:\ERA5_LAND\output_fd_04_drivers\yearly_pentad`
- File driver: `meteorological_drivers_pentad_YYYY.nc`

Jadi **tidak perlu membaca ulang 372 × 3 file bulanan** dan **tidak perlu download ulang P/T2m/D2m**.

## Tujuan

Membuat tabel:

> **1 baris = 1 grid-cell flash-drought start**

dengan karakteristik event + kondisi meteorologi masing-masing event.

Notebook ini **belum mengklasifikasikan** event menjadi X/Y/Z. Itu dilakukan setelah distribusi event-level diperiksa.

In [1]:
from pathlib import Path
import warnings, json, re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("once")

BASE = Path(r"D:\ERA5_LAND")

DETECTION_FILE = (
    BASE / "output_fd_03b_sensitivity"
    / "fd_detection_M2A_1995_2025_indonesia.nc"
)

MET_DIR = (
    BASE / "output_fd_04_drivers"
    / "yearly_pentad"
)

OUTPUT_DIR = (
    BASE / "output_fd_REV1_01_event_level_drivers"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_START = 1996
ANALYSIS_END = 2024

# t-3, t-2, t-1 = antecedent
# t0 hingga t+onset_time = rapid development
LAGS = np.arange(-3, 6)

print("Detection :", DETECTION_FILE)
print("Drivers   :", MET_DIR)
print("Output    :", OUTPUT_DIR)

Detection : D:\ERA5_LAND\output_fd_03b_sensitivity\fd_detection_M2A_1995_2025_indonesia.nc
Drivers   : D:\ERA5_LAND\output_fd_04_drivers\yearly_pentad
Output    : D:\ERA5_LAND\output_fd_REV1_01_event_level_drivers


## 1. Cek file driver dan deteksi nama variabel

Karena nama variabel NetCDF bisa berbeda dari nama file, notebook akan membuka satu file contoh dan mencari otomatis:

- precipitation
- T2m
- VPD

Jika VPD tidak disimpan sebagai file terpisah tetapi terdapat sebagai **variabel di dalam NetCDF**, notebook tetap akan menemukannya.

In [2]:
if not MET_DIR.exists():
    raise FileNotFoundError(f"Folder tidak ditemukan: {MET_DIR}")

files = sorted(MET_DIR.glob("meteorological_drivers_pentad_*.nc"))

print("Jumlah yearly driver files:", len(files))
print("First:", files[0] if files else None)
print("Last :", files[-1] if files else None)

expected_years = set(range(1995, 2026))
available_years = set()

for p in files:
    m = re.search(r"(\d{4})\.nc$", p.name)
    if m:
        available_years.add(int(m.group(1)))

missing_years = sorted(expected_years - available_years)
print("Missing years:", missing_years)

if missing_years:
    raise FileNotFoundError(
        f"Yearly pentad files belum lengkap. Missing: {missing_years}"
    )

sample_file = MET_DIR / "meteorological_drivers_pentad_2000.nc"
ds0 = xr.open_dataset(sample_file)

print("\nSAMPLE DATASET:", sample_file)
print(ds0)
print("\nData variables:")
for v in ds0.data_vars:
    print(
        f"{v:30s}",
        "| dims =", ds0[v].dims,
        "| units =", ds0[v].attrs.get("units", "")
    )

Jumlah yearly driver files: 31
First: D:\ERA5_LAND\output_fd_04_drivers\yearly_pentad\meteorological_drivers_pentad_1995.nc
Last : D:\ERA5_LAND\output_fd_04_drivers\yearly_pentad\meteorological_drivers_pentad_2025.nc
Missing years: []

SAMPLE DATASET: D:\ERA5_LAND\output_fd_04_drivers\yearly_pentad\meteorological_drivers_pentad_2000.nc


<frozen importlib._bootstrap>:488: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject


<xarray.Dataset> Size: 92MB
Dimensions:    (year: 1, pentad: 73, latitude: 171, longitude: 461)
Coordinates:
  * year       (year) int64 8B 2000
  * pentad     (pentad) int16 146B 1 2 3 4 5 6 7 8 9 ... 66 67 68 69 70 71 72 73
  * latitude   (latitude) float64 1kB -11.0 -10.9 -10.8 -10.7 ... 5.8 5.9 6.0
  * longitude  (longitude) float64 4kB 95.0 95.1 95.2 95.3 ... 140.8 140.9 141.0
    number     int64 8B ...
Data variables:
    tp_mm      (year, pentad, latitude, longitude) float32 23MB ...
    t2m_c      (year, pentad, latitude, longitude) float32 23MB ...
    d2m_c      (year, pentad, latitude, longitude) float32 23MB ...
    vpd_kpa    (year, pentad, latitude, longitude) float32 23MB ...
Attributes:
    title:               ERA5-Land pentad meteorological drivers for flash-dr...
    year:                2000
    calendar:            365-day no-leap; Feb 29 removed; 73 pentads of 5 days
    precipitation_mode:  cumulative
    vpd_formula:         0.611*exp(17.5*T/(T+240.978)) - 0.61

In [3]:
def find_var(ds, kind):
    """
    Cari nama variabel secara lebih toleran.
    Cocok untuk nama seperti:
    tp_sum_mm, precip_pentad_mm, t2m_mean_c, vpd_mean_kpa, d2m_mean_c, dll.
    """
    scored = []

    for v in ds.data_vars:
        name = v.lower()
        long_name = str(ds[v].attrs.get("long_name", "")).lower()
        standard_name = str(ds[v].attrs.get("standard_name", "")).lower()
        units = str(ds[v].attrs.get("units", "")).lower()

        text = " ".join([name, long_name, standard_name, units])
        score = 0

        if kind == "precip":
            # nama umum ERA5 / hasil olahan
            if name == "tp":
                score += 30
            if "precip" in name:
                score += 25
            if name.startswith("tp_") or name.endswith("_tp"):
                score += 20
            if "rain" in name:
                score += 10
            if "precip" in text:
                score += 10
            if "total precipitation" in text:
                score += 10

        elif kind == "t2m":
            if name == "t2m":
                score += 30
            if "t2m" in name:
                score += 25
            if "2m_temperature" in name or "2m temperature" in text:
                score += 20
            if "temperature_2m" in name:
                score += 20
            if "air_temperature" in name or "air temperature" in text:
                score += 10

            # jangan sampai dewpoint terpilih sebagai T2m
            if "dew" in text or "d2m" in name:
                score -= 50

        elif kind == "vpd":
            if name == "vpd":
                score += 30
            if "vpd" in name:
                score += 25
            if "vapor_pressure_deficit" in name or "vapour_pressure_deficit" in name:
                score += 20
            if "vapor pressure deficit" in text or "vapour pressure deficit" in text:
                score += 15

        elif kind == "d2m":
            if name == "d2m":
                score += 30
            if "d2m" in name:
                score += 25
            if "dewpoint" in name or "dew_point" in name:
                score += 20
            if "dewpoint" in text or "dew point" in text:
                score += 15

        if score > 0:
            scored.append((score, v))

    if not scored:
        return None

    scored.sort(key=lambda x: (-x[0], x[1]))
    return scored[0][1]


P_VAR = find_var(ds0, "precip")
T_VAR = find_var(ds0, "t2m")
VPD_VAR = find_var(ds0, "vpd")
D2M_VAR = find_var(ds0, "d2m")

print("Detected variables:")
print("  precipitation :", P_VAR)
print("  T2m           :", T_VAR)
print("  VPD           :", VPD_VAR)
print("  D2m           :", D2M_VAR)

print("\nAll data variables in sample file:")
for v in ds0.data_vars:
    print(
        f"  {v:30s}",
        "| units =", ds0[v].attrs.get("units", ""),
        "| long_name =", ds0[v].attrs.get("long_name", "")
    )

if P_VAR is None or T_VAR is None:
    raise KeyError(
        "Masih belum menemukan precipitation dan/atau T2m. "
        "Copy output 'All data variables in sample file' ke ChatGPT."
    )

if VPD_VAR is None and D2M_VAR is None:
    raise KeyError(
        "Tidak ada VPD maupun D2m dalam yearly pentad file. "
        "Copy output 'All data variables in sample file' ke ChatGPT."
    )

Detected variables:
  precipitation : tp_mm
  T2m           : t2m_c
  VPD           : vpd_kpa
  D2m           : d2m_c

All data variables in sample file:
  tp_mm                          | units = mm | long_name = Total precipitation
  t2m_c                          | units = degC | long_name = 2 metre temperature
  d2m_c                          | units = degC | long_name = 2 metre dewpoint temperature
  vpd_kpa                        | units = kPa | long_name = 


### Catatan VPD

Jika `VPD_VAR` ditemukan, notebook memakai VPD yang **sudah dihitung pada workflow driver lama**.

Jika tidak ada VPD tetapi ada T2m dan D2m pentad, VPD dihitung dengan persamaan tekanan uap jenuh:

\[
e_s(T)=0.6108\exp\left(\frac{17.27T}{T+237.3}\right)
\]

\[
VPD=e_s(T)-e_s(T_d)
\]

dengan T dan Td dalam °C, menghasilkan VPD dalam kPa.

In [4]:
def to_celsius(da):
    units = str(da.attrs.get("units", "")).lower()
    # ERA5 umumnya Kelvin; fallback berdasarkan magnitudo juga disediakan.
    if units in {"k", "kelvin"}:
        return da - 273.15

    sample = float(da.isel({d: 0 for d in da.dims}).values)
    if np.isfinite(sample) and sample > 100:
        return da - 273.15

    return da


def calc_vpd_kpa(t2m, d2m):
    T = to_celsius(t2m)
    Td = to_celsius(d2m)

    es = 0.6108 * np.exp((17.27 * T) / (T + 237.3))
    ea = 0.6108 * np.exp((17.27 * Td) / (Td + 237.3))

    vpd = (es - ea).clip(min=0)
    vpd.attrs["units"] = "kPa"
    vpd.attrs["long_name"] = "vapor pressure deficit"
    return vpd

## 2. Load deteksi final dan identifikasi semua event starts

In [5]:
if not DETECTION_FILE.exists():
    raise FileNotFoundError(f"Detection file tidak ditemukan: {DETECTION_FILE}")

det = xr.open_dataset(DETECTION_FILE, mask_and_scale=False)

required = [
    "fd_event_start", "year", "pentad",
    "latitude", "longitude", "valid_grid_mask"
]

missing = [v for v in required if v not in det]
if missing:
    raise KeyError(f"Detection file kehilangan: {missing}")

det_year = np.asarray(det["year"].values).astype(int)
det_pentad = np.asarray(det["pentad"].values).astype(int)

analysis_time = (
    (det["year"] >= ANALYSIS_START)
    & (det["year"] <= ANALYSIS_END)
)

start_mask = (
    (det["fd_event_start"] == 1)
    & analysis_time
    & (det["valid_grid_mask"] == 1)
)

start_np = np.asarray(start_mask.values, dtype=bool)

t_idx, y_idx, x_idx = np.where(start_np)
n_events = len(t_idx)

print("Grid-cell FD starts:", f"{n_events:,}")

EXPECTED = 559_676
if n_events != EXPECTED:
    print(
        f"NOTE: audited run sebelumnya = {EXPECTED:,}; "
        f"file aktif menghasilkan {n_events:,}. "
        "Cek file deteksi jika selisih tidak diharapkan."
    )

lat = np.asarray(det["latitude"].values)
lon = np.asarray(det["longitude"].values)

events = pd.DataFrame({
    "event_id": np.arange(n_events, dtype=np.int64),
    "time_index": t_idx.astype(np.int32),
    "year": det_year[t_idx].astype(np.int16),
    "pentad": det_pentad[t_idx].astype(np.int8),
    "latitude": lat[y_idx].astype(np.float32),
    "longitude": lon[x_idx].astype(np.float32),
    "_y_index": y_idx.astype(np.int32),
    "_x_index": x_idx.astype(np.int32),
})

display(events.head())

Grid-cell FD starts: 559,676


,event_id,time_index,year,pentad,latitude,longitude,_y_index,_x_index
0,0,73,1996,1,-8.2,131.000000,28,360
1,1,73,1996,1,-7.5,110.000000,35,150
2,2,73,1996,1,-7.2,108.599998,38,136
3,3,73,1996,1,-7.2,108.699997,38,137
4,4,73,1996,1,-7.0,111.300003,40,163


## 3. Tambahkan karakteristik FD tiap event

In [6]:
def extract_det_start_var(varname):
    da = det[varname]
    return np.asarray(
        da.values[t_idx, y_idx, x_idx],
        dtype=np.float32
    )


char_map = {
    "onset_time_pentads": "onset_time_pentads_at_start",
    "onset_speed_pp_per_pentad": "onset_speed_at_start",
    "duration_pentads": "duration_pentads_at_start",
    "severity_p40_pp_pentad": "severity_p40_at_start",
    "minimum_rzsm_percentile": "minimum_percentile_at_start",
}

for out_name, src_name in char_map.items():
    if src_name in det:
        events[out_name] = extract_det_start_var(src_name)
    else:
        print("WARNING: missing detection variable:", src_name)

if "onset_time_pentads" not in events:
    raise KeyError("onset_time_pentads diperlukan untuk rapid-development window.")

display(events.head())

,event_id,time_index,year,pentad,latitude,longitude,_y_index,_x_index,onset_time_pentads,onset_speed_pp_per_pentad,duration_pentads,severity_p40_pp_pentad,minimum_rzsm_percentile
0,0,73,1996,1,-8.2,131.000000,28,360,2.0,11.246787,5.0,82.107971,11.439589
1,1,73,1996,1,-7.5,110.000000,35,150,1.0,32.133675,4.0,72.827766,11.439589
2,2,73,1996,1,-7.2,108.599998,38,136,1.0,70.694092,4.0,101.748070,1.799486
3,3,73,1996,1,-7.2,108.699997,38,137,1.0,54.627251,4.0,72.827766,11.439589
4,4,73,1996,1,-7.0,111.300003,40,163,1.0,64.267349,4.0,95.321335,5.012854


## 4. Bangun lookup year–pentad

Agar lag seperti `1997 P01 - 1 pentad` otomatis menjadi `1996 P73`, kita memakai `time_index` deteksi sebagai kalender kontinu.

In [7]:
time_lookup = pd.DataFrame({
    "time_index": np.arange(len(det_year)),
    "year": det_year,
    "pentad": det_pentad,
})

pair_to_t = {
    (int(y), int(p)): int(t)
    for t, y, p in time_lookup[
        ["time_index", "year", "pentad"]
    ].itertuples(index=False, name=None)
}

t_to_pair = {
    int(t): (int(y), int(p))
    for t, y, p in time_lookup[
        ["time_index", "year", "pentad"]
    ].itertuples(index=False, name=None)
}

print("Lookup length:", len(t_to_pair))
print("Example:", list(t_to_pair.items())[:3])

Lookup length: 2263
Example: [(0, (1995, 1)), (1, (1995, 2)), (2, (1995, 3))]


## 5. Fungsi normalisasi yearly pentad file

Notebook mendukung dimensi waktu yang bernama `pentad`, `time`, atau nama lain selama panjangnya sekitar 73 pentad per tahun.

In [8]:
def standardize_spatial_names(ds):
    rename = {}

    if "lat" in ds.dims and "latitude" not in ds.dims:
        rename["lat"] = "latitude"
    if "lon" in ds.dims and "longitude" not in ds.dims:
        rename["lon"] = "longitude"

    if rename:
        ds = ds.rename(rename)

    return ds


def find_pentad_dim(da):
    for d in da.dims:
        if d.lower() == "pentad":
            return d

    for d in da.dims:
        if d.lower() in {"time", "time_index"}:
            return d

    # fallback: dimensi non-spasial dengan ukuran ~73
    for d in da.dims:
        if d not in {"latitude", "longitude"}:
            if 70 <= da.sizes[d] <= 74:
                return d

    raise ValueError(f"Tidak menemukan dimensi pentad pada {da.dims}")


def get_year_dataset(year):
    path = MET_DIR / f"meteorological_drivers_pentad_{year}.nc"

    ds = xr.open_dataset(path)
    ds = standardize_spatial_names(ds)

    # Pastikan grid sama dengan detection.
    if "latitude" not in ds.coords or "longitude" not in ds.coords:
        raise KeyError(f"{path.name}: latitude/longitude tidak ditemukan.")

    if not np.allclose(ds["latitude"].values, det["latitude"].values):
        raise ValueError(f"{path.name}: latitude grid tidak sama.")

    if not np.allclose(ds["longitude"].values, det["longitude"].values):
        raise ValueError(f"{path.name}: longitude grid tidak sama.")

    return ds

## 6. Ekstraksi meteorologi event-level

Untuk setiap event akan dibuat:

- `*_pre3` → rata-rata t−3, t−2, t−1
- `*_lag0` → nilai saat rapid-drying initiation
- `*_rapid` → rata-rata t0 sampai t+onset_time
- `*_delta_rapid_minus_pre` → perubahan dari kondisi antecedent menuju rapid development

Agar RAM aman, proses dilakukan **per tahun target meteorologi**, bukan membuka seluruh 1995–2025 sekaligus.

In [9]:
# Siapkan matriks [event, lag] untuk tiap variabel.
lag_list = list(LAGS)

P_mat = np.full((n_events, len(lag_list)), np.nan, dtype=np.float32)
T_mat = np.full_like(P_mat, np.nan)
VPD_mat = np.full_like(P_mat, np.nan)

# Mapping target (year, pentad) -> daftar event/lag yang memerlukan data itu.
requests = {}

for j, lag in enumerate(lag_list):
    target_t = t_idx + lag
    valid = (target_t >= 0) & (target_t < len(det_year))

    for ev_i in np.where(valid)[0]:
        yy, pp = t_to_pair[int(target_t[ev_i])]
        requests.setdefault(yy, []).append(
            (ev_i, j, pp)
        )

print("Meteorological years requested:", min(requests), "-", max(requests))
print("Number of years:", len(requests))

Meteorological years requested: 1995 - 2025
Number of years: 31


In [10]:
for year in sorted(requests):
    print("Processing", year, "...")

    ds = get_year_dataset(year)

    p_da = ds[P_VAR]
    t_da = ds[T_VAR]

    if VPD_VAR is not None and VPD_VAR in ds:
        vpd_da = ds[VPD_VAR]
    else:
        if D2M_VAR is None or D2M_VAR not in ds:
            ds.close()
            raise KeyError(
                f"{year}: VPD tidak ada dan D2m tidak tersedia."
            )
        vpd_da = calc_vpd_kpa(
            ds[T_VAR],
            ds[D2M_VAR]
        )

    p_dim = find_pentad_dim(p_da)
    t_dim = find_pentad_dim(t_da)
    v_dim = find_pentad_dim(vpd_da)

    # Peta pentad nominal -> positional index.
    # Asumsi yearly file berisi pentad 1..N secara berurutan.
    n_p = p_da.sizes[p_dim]

    # Kelompokkan semua request untuk tahun ini.
    req = requests[year]

    ev_ids = np.array([r[0] for r in req], dtype=int)
    lag_ids = np.array([r[1] for r in req], dtype=int)
    pentads = np.array([r[2] for r in req], dtype=int)

    pos = pentads - 1

    ok = (pos >= 0) & (pos < n_p)
    if not np.all(ok):
        bad = sorted(set(pentads[~ok].tolist()))
        ds.close()
        raise ValueError(
            f"{year}: pentad {bad} tidak ada di yearly file."
        )

    yy_idx = events.loc[ev_ids, "_y_index"].to_numpy(dtype=int)
    xx_idx = events.loc[ev_ids, "_x_index"].to_numpy(dtype=int)

    # Vectorized point indexing.
    evt_dim = "request"

    P_vals = p_da.isel({
        p_dim: xr.DataArray(pos, dims=evt_dim),
        "latitude": xr.DataArray(yy_idx, dims=evt_dim),
        "longitude": xr.DataArray(xx_idx, dims=evt_dim),
    }).values

    T_vals = t_da.isel({
        t_dim: xr.DataArray(pos, dims=evt_dim),
        "latitude": xr.DataArray(yy_idx, dims=evt_dim),
        "longitude": xr.DataArray(xx_idx, dims=evt_dim),
    }).values

    V_vals = vpd_da.isel({
        v_dim: xr.DataArray(pos, dims=evt_dim),
        "latitude": xr.DataArray(yy_idx, dims=evt_dim),
        "longitude": xr.DataArray(xx_idx, dims=evt_dim),
    }).values

    P_mat[ev_ids, lag_ids] = np.asarray(P_vals, dtype=np.float32)
    T_mat[ev_ids, lag_ids] = np.asarray(T_vals, dtype=np.float32)
    VPD_mat[ev_ids, lag_ids] = np.asarray(V_vals, dtype=np.float32)

    ds.close()

print("Extraction complete.")

Processing 1995 ...
Processing 1996 ...
Processing 1997 ...
Processing 1998 ...
Processing 1999 ...
Processing 2000 ...
Processing 2001 ...
Processing 2002 ...
Processing 2003 ...
Processing 2004 ...
Processing 2005 ...
Processing 2006 ...
Processing 2007 ...
Processing 2008 ...
Processing 2009 ...
Processing 2010 ...
Processing 2011 ...
Processing 2012 ...
Processing 2013 ...
Processing 2014 ...
Processing 2015 ...
Processing 2016 ...
Processing 2017 ...
Processing 2018 ...
Processing 2019 ...
Processing 2020 ...
Processing 2021 ...
Processing 2022 ...
Processing 2023 ...
Processing 2024 ...
Processing 2025 ...
Extraction complete.


## 7. Cek units sebelum menghitung anomali/klimatologi

Cell ini hanya audit. Jangan lanjut interpretasi jika precipitation masih dalam meter per pentad atau temperatur masih Kelvin tanpa mengetahui bagaimana workflow lama menyimpannya.

In [11]:
ds_check = get_year_dataset(2000)

for label, var in [
    ("P", P_VAR),
    ("T2m", T_VAR),
    ("VPD", VPD_VAR if VPD_VAR is not None else D2M_VAR),
]:
    if var is not None and var in ds_check:
        print(
            label,
            "| variable =", var,
            "| units =", ds_check[var].attrs.get("units", ""),
            "| long_name =", ds_check[var].attrs.get("long_name", "")
        )

ds_check.close()

print("\nRaw event-level medians:")
print("P   :", np.nanmedian(P_mat))
print("T2m :", np.nanmedian(T_mat))
print("VPD :", np.nanmedian(VPD_mat))

P | variable = tp_mm | units = mm | long_name = Total precipitation
T2m | variable = t2m_c | units = degC | long_name = 2 metre temperature
VPD | variable = vpd_kpa | units = kPa | long_name = 

Raw event-level medians:
P   : 35.004257
T2m : 25.047958
VPD : 0.39937186


## 8. Bentuk ringkasan setiap event

Pada tahap ini kita **belum memaksakan threshold driver**.

Kita hanya menyimpan nilai meteorologi tiap event. Untuk klasifikasi berikutnya, kita akan menghitung anomali relatif terhadap klimatologi/calendar-pentad atau matched control secara konsisten.

In [12]:
onset_time = events["onset_time_pentads"].to_numpy(dtype=float)

def summarize_event_matrix(mat, prefix):
    pre_mask = np.isin(LAGS, [-3, -2, -1])
    lag0_idx = np.where(LAGS == 0)[0][0]

    pre3 = np.nanmean(mat[:, pre_mask], axis=1)
    lag0 = mat[:, lag0_idx]

    rapid = np.full(n_events, np.nan, dtype=np.float32)

    for i in range(n_events):
        mask = (LAGS >= 0) & (LAGS <= onset_time[i])
        rapid[i] = np.nanmean(mat[i, mask])

    events[f"{prefix}_pre3"] = pre3.astype(np.float32)
    events[f"{prefix}_lag0"] = lag0.astype(np.float32)
    events[f"{prefix}_rapid"] = rapid.astype(np.float32)
    events[f"{prefix}_delta_rapid_minus_pre"] = (
        rapid - pre3
    ).astype(np.float32)


summarize_event_matrix(P_mat, "precip")
summarize_event_matrix(T_mat, "t2m")
summarize_event_matrix(VPD_mat, "vpd")

display(
    events[
        [
            "event_id", "year", "pentad",
            "onset_time_pentads",
            "precip_pre3", "precip_rapid",
            "t2m_pre3", "t2m_rapid",
            "vpd_pre3", "vpd_rapid",
        ]
    ].head()
)

,event_id,year,pentad,onset_time_pentads,precip_pre3,precip_rapid,t2m_pre3,t2m_rapid,vpd_pre3,vpd_rapid
0,0,1996,1,2.0,36.096447,20.255415,27.292856,27.364389,0.672773,0.690988
1,1,1996,1,1.0,36.413216,90.915085,21.180351,21.168514,0.265141,0.256161
2,2,1996,1,1.0,64.821861,89.120743,22.748146,22.464638,0.282218,0.268246
3,3,1996,1,1.0,54.977711,77.327621,22.088411,21.704994,0.288346,0.256662
4,4,1996,1,1.0,58.075390,49.788418,25.467026,24.920616,0.456690,0.376225


## 9. Simpan tabel event-level

Kolom internal `_y_index` dan `_x_index` tidak disimpan dalam output final.

In [13]:
save_events = events.drop(
    columns=["_y_index", "_x_index"],
    errors="ignore"
)

parquet_path = OUTPUT_DIR / "event_level_driver_table_RAW.parquet"
csv_path = OUTPUT_DIR / "event_level_driver_table_RAW.csv.gz"

try:
    save_events.to_parquet(parquet_path, index=False)
    saved = parquet_path
except Exception as e:
    print("Parquet unavailable:", e)
    save_events.to_csv(
        csv_path,
        index=False,
        compression="gzip"
    )
    saved = csv_path

qc = save_events.describe(include="all").T
qc.to_csv(
    OUTPUT_DIR / "event_level_driver_RAW_qc.csv"
)

manifest = {
    "analysis_period": [ANALYSIS_START, ANALYSIS_END],
    "detection_file": str(DETECTION_FILE),
    "meteorological_dir": str(MET_DIR),
    "P_VAR": P_VAR,
    "T_VAR": T_VAR,
    "VPD_VAR": VPD_VAR,
    "D2M_VAR": D2M_VAR,
    "n_gridcell_event_starts": int(n_events),
    "row_definition": "one row = one grid-cell M2A event start",
    "classification_done": False,
    "anomaly_standardization_done": False,
    "note": (
        "RAW event-level meteorology. "
        "Do not interpret driver dominance before anomaly/control normalization."
    )
}

(OUTPUT_DIR / "manifest_REV1_01_v2.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("DONE")
print("Rows :", f"{len(save_events):,}")
print("Saved:", saved)

DONE
Rows : 559,676
Saved: D:\ERA5_LAND\output_fd_REV1_01_event_level_drivers\event_level_driver_table_RAW.parquet


# STOP DI SINI

Kirim ke saya tiga output berikut setelah notebook berhasil:

1. hasil **Detected variables**;
2. hasil **units + raw event-level medians**;
3. baris terakhir `DONE / Rows / Saved`.

Setelah itu kita tentukan cara normalisasi yang paling konsisten dengan matched-control lama, lalu baru membuat **REV1-02 — Driver Classification (X/Y/Z)**.

**Jangan klasifikasikan berdasarkan nilai raw terlebih dahulu.**